# 新規100候補：抽出・検証・品質評価

**90候補を実装、10候補は役・点数・完全振聴判定が未対応です。全100候補の正確性が確認済みという意味ではありません。**
実牌譜の検証はこのNotebookで行います。モデル学習はありません。既存20＋3特徴量、C1〜C10、H1〜H6は変更しません。

最初はN1→N2（A）→N3→N4→N5。監査を確認後、N2でBへ変更してN3〜N5。全件完了後にN6のCを実行します。
段位コード16以下／18以上、17除外、初期1500対局と同じ時間帯の除外、共通対局除外を維持します。


## N1：Drive接続・固定版取得・合成テスト

依存ライブラリは別フォルダ・別プロセスで使用します。実牌譜はここでは読みません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, os, subprocess, json, urllib.request, hashlib
CODE_COMMIT = 'd6b112589536cdf94f78e59993c39fb22da1b1d8'
SOURCE = Path('/tmp') / ('features_100_' + CODE_COMMIT[:12])
FILES = ['choices/choices.py', 'decision_features/__init__.py', 'decision_features/definitions.py', 'decision_features/extract.py', 'decision_features/pipeline.py', 'feature_design_100/feature_candidates_100.csv', 'features_100/__init__.py', 'features_100/calls.py', 'features_100/cli.py', 'features_100/extract.py', 'features_100/options.py', 'features_100/pipeline.py', 'features_100/quality.py', 'features_100/requirements.txt', 'features_100/rules.py', 'features_100/shapes.py', 'features_100/tests/__init__.py', 'features_100/tests/test_core.py', 'features_100/tests/test_integration.py', 'features_100/tests/test_quality.py', 'features_100/tests/test_rule_tables.py', 'mahjong_research/__init__.py', 'mahjong_research/analysis.py', 'mahjong_research/inputs.py', 'mahjong_research/legacy.py', 'mahjong_research/tenhou.py', 'smoothing/study.py']
for relative in FILES:
    target = SOURCE / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    url = f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{CODE_COMMIT}/research_v2/{relative}'
    content = urllib.request.urlopen(url).read()
    if target.exists() and target.read_bytes() != content: raise RuntimeError('固定版と不一致: ' + relative)
    if not target.exists(): target.write_bytes(content)
DEPS = SOURCE / '_dependencies'
stamp = DEPS / 'install_complete.txt'
requirements = SOURCE / 'features_100/requirements.txt'
fingerprint = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(requirements)])
    stamp.write_text(fingerprint)
elif stamp.read_text() != fingerprint: raise RuntimeError('依存版が不一致。新しいセッションを使用してください。')
ENV = dict(os.environ, PYTHONPATH=os.pathsep.join(map(str,[DEPS,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])), PYTHONIOENCODING='utf-8', MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
# 日本語図用フォント。解析結果には影響しません。
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定版取得・合成テスト完了。実牌譜の検証はN4以降です。')


## N2：入力・モード・出力の設定

RUN_DIRはinput_plan.jsonとshared_games.csvがある既存実行フォルダ。C5キャッシュの場所は計画ファイルから読みます。
元ZIPが移動していればZIP_OVERRIDEを変更してください。PLAYERは完全一致名です（全角ｎ）。
Aは各群1〜10対局、または明示ID最大60件。Bは全C5対局を抽出し、17・初期1500等の除外は窓作成時に適用します。
新出力の日時フォルダは毎回作成します。同じ版・元XML・対象の成功キャッシュだけ再利用します。


In [ ]:
ROOT = Path('/content/drive/MyDrive/卒論')
RUN_DIR = ROOT / 'research_v2_runs/20261105T071722Z_5da6c9ad'
OUTPUT_ROOT = ROOT / 'research_v2_features_100'
PLAYER = 'しょぼんｎ'
KEY = 'shobon'
ZIP_OVERRIDE = None
MODE = 'A'  # 最初はA。監査確認後にBへ変更。
PILOT_PER_GROUP = 3
PILOT_IDS = None  # 指定するならC5のgame_idを重複なく列挙
EXPLORATION_END_HOUR = ''  # C実行前に例 '2014-01-01 00:00:00' のように正時の境界を指定。例は研究上の推薦ではない。
CONFIG = SOURCE / 'config.json'
RECEIPT = SOURCE / ('preview_' + MODE + '.json')
def save_config():
    CONFIG.write_text(json.dumps(dict(run_dir=str(RUN_DIR),output_root=str(OUTPUT_ROOT),player=PLAYER,key=KEY,
        zip_override=ZIP_OVERRIDE,pilot_per_group=PILOT_PER_GROUP,pilot_ids=PILOT_IDS,
        exploration_end_hour=EXPLORATION_END_HOUR),ensure_ascii=False),encoding='utf8')
def run_step(action,extra=()):
    subprocess.check_call([sys.executable,'-m','features_100.cli',action,str(CONFIG),'--mode',MODE,
        '--receipt',str(RECEIPT),*extra],env=ENV)
save_config()
print('入力:',RUN_DIR,'\n出力親:',OUTPUT_ROOT,'\nモード:',MODE)


## N3：入力監査と処理件数の表示（抽出なし）

C5署名・ハッシュ・ID集合・ZIP内の対象名を確認します。表示件数を確認してN4へ進みます。

In [ ]:
run_step('preview')


## N4：Aの小規模検証／Bの全件抽出

全実装済み候補を共通の状態復元から計算します。初回は時間がかかるためAの実測秒数を確認してください。
中断後は同じ固定版と設定でN1〜N4を再実行。成功キャッシュは再利用し、失敗対局は監査に残します。
Aは判断直前の状態・X/Y・実行・理由をtracesに保存します。BキャッシュからAの詳細監査を追加する場合だけ指定対局を読み直します。


In [ ]:
run_step('extract')
EXTRACTION_DIR = Path(json.loads(RECEIPT.with_suffix('.result.json').read_text())['output'])
print('抽出結果:', EXTRACTION_DIR)


## N5：監査・分子分母の確認

保留10候補はunimplemented、機会ゼロはno_opportunity、不明条件を含む場合はunknown。
unknownの観測済みカウンタは部分集計であり、完成した割合として使用しません。
各候補の正例が出ない場合は未検証です。Aの少数対局を詰めて10/20対局窓を作りません。


In [ ]:
import pandas as pd
from IPython.display import display, Markdown
for name in ['implementation_status.csv','extraction_audit.csv','feature_audit.csv']:
    print(name)
    display(pd.read_csv(EXTRACTION_DIR/name))
display(Markdown((EXTRACTION_DIR/'report_ja.md').read_text(encoding='utf8')))
print('Aの判断記録:', EXTRACTION_DIR/'traces')


## N6：C・保存済み分子分母から品質評価（ZIPを読まない）

全件完了したBのフォルダを指定します。探索期間末を事前に決め、以降の窓は特徴量統計から除外します。
過去に確認済みのデータを「未使用」に戻すことはできません。10/20対局は時間帯を分割せず、同一段位群・非重複で作成。
原対局が共通の感度分析も出力します。比率は分子合計÷分母合計、窓を等重みで比較します。
親比較の対応状況はparent_coverage.csvを参照してください。全100候補の親比較は未完です。


In [ ]:
# セッション再開時は、Bで出力された実在フォルダを指定してください。
FULL_EXTRACTION_DIR = EXTRACTION_DIR
if MODE != 'B':
    print('AではCを実行しません。監査後にMODE=BでN2〜N5を実行してください。')
else:
    if not EXPLORATION_END_HOUR:
        raise ValueError('N2のEXPLORATION_END_HOURを設定してください。未使用評価期間を残す境界が必要です。')
    save_config()
    run_step('quality',['--extraction-dir',str(FULL_EXTRACTION_DIR)])
    QUALITY_DIR = Path(json.loads(RECEIPT.with_suffix('.result.json').read_text())['output'])
    print('品質評価の保存先:',QUALITY_DIR)
    display(pd.read_csv(QUALITY_DIR/'feature_comparison.csv'))
    display(Markdown((QUALITY_DIR/'report_ja.md').read_text(encoding='utf8')))
